In [8]:
import pandas as pd
import numpy as np

# ==========================
# Load Data
# ==========================

df = pd.read_csv(r"D:\DATA Science\messy_ecommerce_sales_data.csv")

# Remove duplicates
df = df.drop_duplicates()

# Clean column names
df.columns = df.columns.str.strip()

# ==========================
# Data Quality Checks
# ==========================

# Temporary conversion to detect invalid price formats
temp_price = pd.to_numeric(
    df["Price"],
    errors="coerce"
)

# Invalid price format flag
invalid_price = (
    temp_price.isna() &
    df["Price"].notna()
)

# Convert Quantity
df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
).fillna(0).round()

# Convert Price
df["Price"] = temp_price.fillna(0).round()

# Total Revenue Column
df["Total"] = df["Quantity"] * df["Price"]

# Convert Date
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="mixed",
    errors="coerce"
)

# Clean Category
df["Category"] = (
    df["Category"]
    .replace(r"^\s*$", "Unknown", regex=True)
    .fillna("Unknown")
)

df["Category"]=df["Category"].replace(
    {"electronics":"Electronics",
     "electroncs":"Electronics",
     "ELECtronics":"Electronics",
     "ELECTRONICS":"Electronics",
     "electronic":"Electronics"
    }
)                                     

# ==========================
# Date Features
# ==========================

df["Month"] = df["Order_Date"].dt.month_name()
df["Month_number"] = df["Order_Date"].dt.month
df["Weekday"] = df["Order_Date"].dt.day_name()
df["Weekday_number"]=df["Order_Date"].dt.weekday + 1

# ==========================
# Anomaly Detection
# ==========================

df["Final_Status"] = np.where(
    (df["Quantity"] <= 0) |
    (df["Price"] <= 0) |
    (invalid_price),
    "anomaly",
    "valid"
)

# ==========================
# Business Valid Orders
# ==========================

valid_orders = df.loc[
    (~df["Status"].isin(["Cancelled", "Returned"])) &
    (df["Final_Status"] == "valid")
]

# ==========================
# KPI Calculations
# ==========================

Total_revenue = valid_orders["Total"].sum()

Average_order_value = valid_orders["Total"].mean()

Top_categories = (
    valid_orders
    .groupby("Category")["Total"]
    .sum()
    .sort_values(ascending=False)
)

Top_product = (
    valid_orders
    .groupby("Product")["Total"]
    .sum()
    .sort_values(ascending=False)
)

# ==========================
# Customer Analysis
# ==========================

Customers = df.groupby("Customer_Name").size()

One_time_Customers = (Customers == 1).sum()

Repeated_Customers = (Customers > 1).sum()

Revenue_by_customers = (
    valid_orders
    .groupby("Customer_Name")["Total"]
    .sum()
)

Top_customers = Revenue_by_customers.sort_values(
    ascending=False
)

Customer_segment = Revenue_by_customers.apply(
    lambda x:
    "High_spender" if x >= 1000
    else "Medium_spender" if x >= 500
    else "Low_spender"
)

# ==========================
# Trend Analysis
# ==========================

monthly_trend = (
    valid_orders
    .groupby("Month_number")["Total"]
    .sum()
)

Weekly_revenue = (
    valid_orders
    .groupby("Weekday")["Total"]
    .sum()
)

Weekday_revenue = Weekly_revenue[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
].sum()

Weekend_revenue = Weekly_revenue[
    ["Saturday", "Sunday"]
].sum()

df["Day_Type"] = np.where(
    df["Weekday"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday"
)

# ==========================
# Other KPIs
# ==========================

Cancellation_return_rate = (
    df["Status"]
    .isin(["Cancelled", "Returned"])
    .sum()
    / len(df)
) * 100

Low_performing_categories = (
    Top_categories
    .sort_values()
    .head(5)
)

revenue_distribution_by_categories = (
    (Top_categories / Total_revenue) * 100
).round(2)

revenue_contribution_by_week = (
    (Weekly_revenue / Total_revenue) * 100
).round(2)

df.to_csv("Cleaned_messy_data.csv",index=False)

# ==========================
# Data Quality Findings
# ==========================

print("===== DATA QUALITY =====")
print(f"Total Records          : {len(df)}")
print(f"Anomaly Records        : {(df['Final_Status']=='anomaly').sum()}")
print(f"Duplicate Records Removed : 1")
print()

# ==========================
# KPI Output
# ==========================

print("===== KPI =====\n")

print(f"Total Revenue:\n{Total_revenue}\n")

print(f"Average Order Value:\n{Average_order_value}\n")

print("Top Categories:")
print(Top_categories)
print()

print("===== CUSTOMER ANALYSIS =====\n")

print(f"One-time Customers:\n{One_time_Customers}\n")

print(f"Repeated Customers:\n{Repeated_Customers}\n")

print("Top Customers:")
print(Top_customers.head(5))
print()

print(f"Cancellation / Return Rate: {Cancellation_return_rate:.2f}%")

===== DATA QUALITY =====
Total Records          : 102
Anomaly Records        : 17
Duplicate Records Removed : 1

===== KPI =====

Total Revenue:
71845.0

Average Order Value:
1466.2244897959183

Top Categories:
Category
Clothing       19605.0
Sports         13041.0
Home           12393.0
Books          11444.0
Electronics     8564.0
Unknown         6798.0
Name: Total, dtype: float64

===== CUSTOMER ANALYSIS =====

One-time Customers:
98

Repeated Customers:
2

Top Customers:
Customer_Name
Customer_142    6450.0
Customer_144    4245.0
Customer_170    3710.0
Customer_131    3085.0
Customer_125    3064.0
Name: Total, dtype: float64

Cancellation / Return Rate: 41.18%
